In [1]:
from sqlalchemy import create_engine, MetaData

url = "escola.db"
engine = create_engine(f"sqlite:///{url}")

In [3]:
md_obj = MetaData()
md_obj.reflect(engine)
md_obj.tables.keys()

dict_keys(['Alunos', 'Disciplinas', 'Professores'])

In [4]:
import pandas as pd
pd.read_sql_table("Alunos", engine)

,ID_Aluno,Nome,Email,Rua_Numero,Estado,Nota_Final
0,1,Aluno 1,aluno1@email.com,"Rua 1, 10",BA,5.2
1,2,Aluno 2,aluno2@email.com,"Rua 2, 20",BA,7.2
2,3,Aluno 3,aluno3@email.com,"Rua 3, 30",BA,9.8
3,4,Aluno 4,aluno4@email.com,"Rua 4, 40",SP,8.0
4,5,Aluno 5,aluno5@email.com,"Rua 5, 50",SP,9.7
5,6,Aluno 6,aluno6@email.com,"Rua 6, 60",RJ,9.8
6,7,Aluno 7,aluno7@email.com,"Rua 7, 70",RJ,9.0
7,8,Aluno 8,aluno8@email.com,"Rua 8, 80",MG,5.2
8,9,Aluno 9,aluno9@email.com,"Rua 9, 90",MG,9.9
9,10,Aluno 10,aluno10@email.com,"Rua 10, 100",RJ,8.3


In [5]:
import pandas as pd
pd.read_sql_table("Disciplinas", engine)

,ID_Disciplina,Nome,Descricao,Carga_Horaria,ID_Professor
0,1,Disciplina 1,Descrição da Disciplina 1,60,9
1,2,Disciplina 2,Descrição da Disciplina 2,30,2
2,3,Disciplina 3,Descrição da Disciplina 3,45,2
3,4,Disciplina 4,Descrição da Disciplina 4,90,3
4,5,Disciplina 5,Descrição da Disciplina 5,30,15
5,6,Disciplina 6,Descrição da Disciplina 6,90,9
6,7,Disciplina 7,Descrição da Disciplina 7,60,2
7,8,Disciplina 8,Descrição da Disciplina 8,45,3
8,9,Disciplina 9,Descrição da Disciplina 9,60,1
9,10,Disciplina 10,Descrição da Disciplina 10,45,17


### Configuração LlamaIndex

In [6]:
from dotenv import load_dotenv
import os

_ = load_dotenv()
GROQ_API = os.environ.get("GROQ_API_KEY")

In [7]:
model = "llama-3.3-70b-versatile"
model_hf_emb = "BAAI/bge-m3"

In [8]:
from llama_index.core import Settings
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = Groq(model=model, api_key=GROQ_API)
Settings.embed_model = HuggingFaceEmbedding(model_name=model_hf_emb)

### Configuração BD no Modelo

In [9]:
from llama_index.core import SQLDatabase
from llama_index.core.objects import SQLTableNodeMapping

sql_db = SQLDatabase(engine)
table_node_map = SQLTableNodeMapping(sql_db)

In [11]:
from llama_index.core.objects import SQLTableSchema

table_schema_objs = []
for table_name in md_obj.tables.keys():
    table_schema_objs.append(SQLTableSchema(table_name=table_name))

In [12]:
from llama_index.core.objects import ObjectIndex
from llama_index.core import VectorStoreIndex

obj_index = ObjectIndex.from_objects(
    table_schema_objs,
    table_node_map,
    VectorStoreIndex
)

In [13]:
obj_retriever = obj_index.as_retriever(similarity_top_k=1)

### Testando a Primeira Consulta

In [14]:
from llama_index.core.indices.struct_store.sql_query import SQLTableRetrieverQueryEngine

query_engine = SQLTableRetrieverQueryEngine(
    sql_db,
    obj_retriever
)

In [15]:
response = query_engine.query("Quais estados mais frequentes na tabela Alunos?")

In [16]:
print(response)

Os estados mais frequentes na tabela Alunos são RJ, com 6 ocorrências, seguido por BA, com 5 ocorrências, e SP e MG, com 4 ocorrências cada. O estado RS é o menos frequente, com apenas 1 ocorrência.


In [17]:
response.metadata

{'565c2437-1213-4e83-bddb-f1906397ad68': {'sql_query': 'SELECT Estado, COUNT(ID_Aluno) AS Frequencia FROM Alunos GROUP BY Estado ORDER BY Frequencia DESC',
  'result': [('RJ', 6), ('BA', 5), ('SP', 4), ('MG', 4), ('RS', 1)],
  'col_keys': ['Estado', 'Frequencia']},
 'sql_query': 'SELECT Estado, COUNT(ID_Aluno) AS Frequencia FROM Alunos GROUP BY Estado ORDER BY Frequencia DESC',
 'result': [('RJ', 6), ('BA', 5), ('SP', 4), ('MG', 4), ('RS', 1)],
 'col_keys': ['Estado', 'Frequencia']}

In [18]:
response = query_engine.query("Quais professores residem na cidade de São Paulo?")

In [19]:
print(response)

Os professores que residem na cidade de São Paulo são: 

- Professor 7, que mora na Rua 7, no Bairro 7.
- Professor 8, que mora na Rua 8, no Bairro 8.
- Professor 10, que mora na Rua 10, no Bairro 10.
- Professor 11, que mora na Rua 11, no Bairro 11.
- Professor 20, que mora na Rua 20, no Bairro 20.


In [20]:
response.metadata

{'9a987c45-87ce-4497-93e0-0075b39b34db': {'sql_query': "SELECT Nome, Rua, Bairro FROM Professores WHERE Cidade = 'São Paulo'",
  'result': [('Professor 7', 'Rua 7', 'Bairro 7'),
   ('Professor 8', 'Rua 8', 'Bairro 8'),
   ('Professor 10', 'Rua 10', 'Bairro 10'),
   ('Professor 11', 'Rua 11', 'Bairro 11'),
   ('Professor 20', 'Rua 20', 'Bairro 20')],
  'col_keys': ['Nome', 'Rua', 'Bairro']},
 'sql_query': "SELECT Nome, Rua, Bairro FROM Professores WHERE Cidade = 'São Paulo'",
 'result': [('Professor 7', 'Rua 7', 'Bairro 7'),
  ('Professor 8', 'Rua 8', 'Bairro 8'),
  ('Professor 10', 'Rua 10', 'Bairro 10'),
  ('Professor 11', 'Rua 11', 'Bairro 11'),
  ('Professor 20', 'Rua 20', 'Bairro 20')],
 'col_keys': ['Nome', 'Rua', 'Bairro']}

### Melhorando o Contexto do Modelo

In [22]:
def generate_description_table(table_name, df_amostra):
    prompt = f"""
    Analise a amostra da tabela '{table_name}' abaixo e forneça uma
    curta e breve descrição do conteúdo dessa tabela.
    Informe até o máximo de 5 valores únicos de cada coluna.
    
    Amostra da tabela:
    {df_amostra}
    
    Descrição:
    
    """
    llm = Groq(model=model, api_key=GROQ_API)
    response = llm.complete(prompt=prompt)
    return response.text
    

In [23]:
names_tables = md_obj.tables.keys()
dict_tables = {}
for name_table in names_tables:
    df = pd.read_sql_table(name_table, engine)
    df_amostra = df.head(5).to_string()
    
    descricao = generate_description_table(name_table, df_amostra)
    dict_tables[name_table] = descricao
    print(f"Tabela: {name_table}\nDescrição: {descricao}")
    print("-"*15)


Tabela: Alunos
Descrição: A tabela 'Alunos' contém informações sobre os alunos de uma instituição de ensino. As colunas incluem:

- **ID_Aluno**: Identificador único do aluno, com valores únicos como 1, 2, 3, 4 e 5.
- **Nome**: Nome do aluno, com valores únicos como Aluno 1, Aluno 2, Aluno 3, Aluno 4 e Aluno 5.
- **Email**: Endereço de e-mail do aluno, com valores únicos como aluno1@email.com, aluno2@email.com, aluno3@email.com, aluno4@email.com e aluno5@email.com.
- **Rua_Numero**: Endereço do aluno, com valores únicos como Rua 1, 10, Rua 2, 20, Rua 3, 30, Rua 4, 40 e Rua 5, 50.
- **Estado**: Estado onde o aluno reside, com valores únicos como BA (Bahia) e SP (São Paulo).
- **Nota_Final**: Nota final do aluno, com valores únicos como 5.2, 7.2, 9.8, 8.0 e 9.7.

Essa tabela parece conter informações básicas sobre os alunos, incluindo seus dados de contato, endereço e desempenho acadêmico.
---------------
Tabela: Disciplinas
Descrição: A tabela 'Disciplinas' contém informações sobre disc

In [24]:
table_schema_objs = [
    SQLTableSchema(
        table_name=table_name,
        context_str=dict_tables[table_name]
    )
    for table_name in names_tables
]

In [25]:
table_schema_objs

[SQLTableSchema(table_name='Alunos', context_str="A tabela 'Alunos' contém informações sobre os alunos de uma instituição de ensino. As colunas incluem:\n\n- **ID_Aluno**: Identificador único do aluno, com valores únicos como 1, 2, 3, 4 e 5.\n- **Nome**: Nome do aluno, com valores únicos como Aluno 1, Aluno 2, Aluno 3, Aluno 4 e Aluno 5.\n- **Email**: Endereço de e-mail do aluno, com valores únicos como aluno1@email.com, aluno2@email.com, aluno3@email.com, aluno4@email.com e aluno5@email.com.\n- **Rua_Numero**: Endereço do aluno, com valores únicos como Rua 1, 10, Rua 2, 20, Rua 3, 30, Rua 4, 40 e Rua 5, 50.\n- **Estado**: Estado onde o aluno reside, com valores únicos como BA (Bahia) e SP (São Paulo).\n- **Nota_Final**: Nota final do aluno, com valores únicos como 5.2, 7.2, 9.8, 8.0 e 9.7.\n\nEssa tabela parece conter informações básicas sobre os alunos, incluindo seus dados de contato, endereço e desempenho acadêmico."),
 SQLTableSchema(table_name='Disciplinas', context_str="A tabela

In [26]:
obj_index = ObjectIndex.from_objects(
    table_schema_objs,
    table_node_map,
    VectorStoreIndex
)
obj_retriever = obj_index.as_retriever(similarity_top_k=1)

In [27]:
query_engine_2 = SQLTableRetrieverQueryEngine(sql_db, obj_retriever)
response = query_engine_2.query("Quantos professores de Física temos?")

In [28]:
print(response)

Temos 6 professores de Física.


In [29]:
response.metadata

{'33b32649-52a7-46e3-b5af-4a3643ee1ddd': {'sql_query': "SELECT COUNT(ID_Professor) FROM Professores WHERE Especialidade = 'Física'",
  'result': [(6,)],
  'col_keys': ['COUNT(ID_Professor)']},
 'sql_query': "SELECT COUNT(ID_Professor) FROM Professores WHERE Especialidade = 'Física'",
 'result': [(6,)],
 'col_keys': ['COUNT(ID_Professor)']}

In [30]:
response = query_engine_2.query("Quantos professores de Física que residem em São Paulo?")

In [31]:
print(response)

Existem 2 professores de Física que residem em São Paulo.


In [32]:
response.metadata

{'2c82b46d-331f-4534-b78b-80bb9a0583a6': {'sql_query': "SELECT COUNT(P.ID_Professor) FROM Professores P WHERE P.Especialidade = 'Física' AND P.Cidade = 'São Paulo'",
  'result': [(2,)],
  'col_keys': ['COUNT(P.ID_Professor)']},
 'sql_query': "SELECT COUNT(P.ID_Professor) FROM Professores P WHERE P.Especialidade = 'Física' AND P.Cidade = 'São Paulo'",
 'result': [(2,)],
 'col_keys': ['COUNT(P.ID_Professor)']}

In [36]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(repo_id="BAAI/bge-m3", filename="pytorch_model.bin")
print(model_path)

C:\Users\rodri\.cache\huggingface\hub\models--BAAI--bge-m3\snapshots\5617a9f61b028005a4858fdac845db406aefb181\pytorch_model.bin
